# 06 Test Inference & Ensemble

This notebook orchestrates Stage 06 of the PHM Bearing RUL pipeline.

Pipeline
--------
```
Model Bundle
+
Model-ready Test Dataset
↓
Test Endpoint Extraction
↓
Per-LOBO Test Prediction
↓
Raw Ensemble
↓
Optional Model-specific Calibration
↓
Calibrated Ensemble
↓
RUL_norm → RUL_sec Reconstruction
↓
Optional Test Evaluation
↓
Comparison / Submission Artifact
```

Stage 06 principles
-------------------
- ModelBundle is the sole source of model definition.
- Test does not use Prefix Definition.
- Test does not generate targets.
- Test inference is performed at the final observed endpoint
  of each bearing.
- One endpoint is extracted for each Test bearing.
- Raw predictions are preserved.
- Calibration is optional and uses Stage 05 parameters when available.
- Calibration is applied per LOBO model before ensemble.
- Raw and calibrated results are both preserved for comparison.
- Submission output contains only the selected prediction.


In [ ]:
# =============================================================================
# 0. Imports & Configuration
# =============================================================================

from pathlib import Path
from dataclasses import dataclass, field

import joblib
import pandas as pd
import numpy as np
import json

from utils.bearing_utils import (
    META_COLUMNS, reconstruct_test_rul_sec_from_norm
)
from utils.model_utils import (
    ModelBundle,
    extract_test_endpoints,
    predict_selected_lobo_test_endpoints,
    apply_rul_calibration,
)
from utils.project_utils import assert_required_columns
from utils.ensemble_utils import (
    EnsembleConfig, ensemble_predictions,
)
from utils.eval_utils import evaluate_prediction_table

# -------------------------------------------------------------------------
# Model Artifacts
# -------------------------------------------------------------------------
MODEL_PATHS = {
    "RandomForest": {
        "model_bundle": Path(
            "outputs/05_modeling/RandomForest/model_bundle.pkl"
        ),
        "calibration": Path(
            "outputs/05_modeling/RandomForest/calibration_results.json"
        ),
    },
    "CatBoost": {
        "model_bundle": Path(
            "outputs/05_modeling/CatBoost/model_bundle.pkl"
        ),
        "calibration": Path(
            "outputs/05_modeling/CatBoost/calibration_results.json"
        ),
    },
}

# -------------------------------------------------------------------------
# Runtime Configuration
# -------------------------------------------------------------------------
ensemble_method = "harmonic"
ensemble_weights = None

# Used when calibration_results.json contains parameters for multiple
# calibration methods (for example, when Stage 05 used "both").
CALIBRATION_METHOD = "rational"

# Select which final prediction is written to the submission artifact.
# If calibration is requested but unavailable, the notebook falls back
# to the raw prediction.
SUBMISSION_USE_CALIBRATION = True

TEST_DATASET_PATH = Path("outputs/04_dataset_builder/model_ready_test_dataset.csv")

OUTPUT_DIR = Path("outputs/06_test_inference")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOBO_PREDICTIONS_PATH = OUTPUT_DIR / "lobo_test_predictions.csv"
COMPARISON_PREDICTIONS_PATH = OUTPUT_DIR / "test_rul_comparison.csv"
TEST_METRICS_PATH = OUTPUT_DIR / "test_metrics.json"
ENSEMBLE_CONFIG_PATH = OUTPUT_DIR / "ensemble_config.json"

# 제출 양식 산출물. 대표성을 띄므로 root 폴더에 저장.
SUBMISSION_PATH = Path("submission.csv")

print("=" * 60)
print("Stage 06 — Test Inference & Ensemble")
print("=" * 60)

print(f"Test Dataset       : {TEST_DATASET_PATH}")
print(f"Model Bundles      : {list(MODEL_PATHS)}")
print(f"Ensemble method    : {ensemble_method}")
print(f"Calibration method : {CALIBRATION_METHOD}")
print(f"Use calibration    : {SUBMISSION_USE_CALIBRATION}")

In [ ]:
for algorithm, paths in MODEL_PATHS.items():
    print(f"{algorithm}")
    print(f"Model Bundle={paths['model_bundle']}")
    print(f"Calibration={paths['calibration']}")
    print()

In [ ]:
@dataclass
class ModelSelectionConfig:
    """
    Select the algorithm used for each LOBO held-out bearing.

    default_algorithm
        Algorithm used unless explicitly overridden.

    overrides
        Mapping:
            held_out_bearing -> algorithm
    """

    default_algorithm: str = "RandomForest"
    overrides: dict[str, str] = field(default_factory=dict)

    def algorithm_for(self, held_out_bearing: str) -> str:
        return self.overrides.get(
            held_out_bearing,
            self.default_algorithm,
        )

MODEL_SELECTION = ModelSelectionConfig(
    default_algorithm="RandomForest",
    overrides={
        "Bearing1_1": "CatBoost",
    },
)

In [ ]:
# =============================================================================
# 1. Load Model Bundles
# =============================================================================

model_bundles = {}

for algorithm, paths in MODEL_PATHS.items():
    model_bundle_path = paths["model_bundle"]

    if not model_bundle_path.exists():
        raise FileNotFoundError(
            f"Model Bundle not found for {algorithm}: "
            f"{model_bundle_path}"
        )

    bundle = joblib.load(model_bundle_path)

    if not isinstance(bundle, ModelBundle):
        raise TypeError(
            f"Loaded object is not a valid ModelBundle: {algorithm}"
        )

    if len(bundle.models) != len(bundle.fold_metadata):
        raise ValueError(
            f"ModelBundle invariant violated for {algorithm}: "
            "len(models) must equal len(fold_metadata)."
        )

    if not bundle.models:
        raise ValueError(
            f"ModelBundle contains no trained models: {algorithm}"
        )

    model_bundles[algorithm] = bundle

print("Model Bundles loaded successfully.")

for algorithm, bundle in model_bundles.items():
    print(
        f"{algorithm:12s} | "
        f"Target={bundle.target_column} | "
        f"Features={len(bundle.feature_columns)} | "
        f"LOBO models={len(bundle.models)}"
    )

In [ ]:
# -------------------------------------------------------------------------
# Cross-Bundle Contract Check
# -------------------------------------------------------------------------

reference_algorithm = next(iter(model_bundles))
reference_bundle = model_bundles[reference_algorithm]

reference_fold_bearings = [
    fold.held_out_bearing
    for fold in reference_bundle.fold_metadata
]

for algorithm, bundle in model_bundles.items():
    fold_bearings = [
        fold.held_out_bearing
        for fold in bundle.fold_metadata
    ]

    if fold_bearings != reference_fold_bearings:
        raise RuntimeError(
            "LOBO fold ordering mismatch: "
            f"{reference_algorithm} vs {algorithm}"
        )

    if bundle.feature_columns != reference_bundle.feature_columns:
        raise RuntimeError(
            "Feature schema mismatch: "
            f"{reference_algorithm} vs {algorithm}"
        )

    if bundle.target_column != reference_bundle.target_column:
        raise RuntimeError(
            "Target schema mismatch: "
            f"{reference_algorithm} vs {algorithm}"
        )

print("Cross-bundle schema       : OK")
print("LOBO fold correspondence : OK")

In [ ]:
# =============================================================================
# 2. Load Model-ready Test Dataset
# =============================================================================

if not TEST_DATASET_PATH.exists():
    raise FileNotFoundError(f"Test dataset not found: {TEST_DATASET_PATH}")

test_df = pd.read_csv(TEST_DATASET_PATH)

print("Model-ready Test Dataset loaded.")
print(f"Rows    : {len(test_df):,}")
print(f"Columns : {len(test_df.columns):,}")

display(test_df.head())

In [ ]:
# =============================================================================
# 3. Phase 0 — Input Contract Checkpoint
# =============================================================================

print("=" * 60)
print("Phase 0 — Input Contract Checkpoint")
print("=" * 60)

# -------------------------------------------------------------------------
# Test Metadata
# -------------------------------------------------------------------------
assert_required_columns(test_df, META_COLUMNS)

# -------------------------------------------------------------------------
# Model Features
# -------------------------------------------------------------------------
assert_required_columns(test_df, reference_bundle.feature_columns)

# -------------------------------------------------------------------------
# Target must not exist in Test
# -------------------------------------------------------------------------
if reference_bundle.target_column in test_df.columns:
    raise ValueError(
        "Test dataset unexpectedly contains the model target: "
        f"{reference_bundle.target_column!r}"
    )

# -------------------------------------------------------------------------
# Basic Dataset Validation
# -------------------------------------------------------------------------
if test_df.empty:
    raise ValueError("model_ready_test_dataset must not be empty.")

if test_df["bearing"].isna().any():
    raise ValueError("Test dataset contains missing bearing values.")

if not pd.api.types.is_numeric_dtype(test_df["elapsed_sec"]):
    raise TypeError("'elapsed_sec' must be numeric.")

print("Required metadata columns : OK")
print("Model feature columns     : OK")
print("Target absence             : OK")
print("Dataset non-empty          : OK")
print("Phase 0 checkpoint passed.")

In [ ]:
# =============================================================================
# 4. Phase 1 — Extract Test Endpoints
# =============================================================================

print("=" * 60)
print("Phase 1 — Test Endpoint Extraction")
print("=" * 60)

test_endpoints = extract_test_endpoints(
    dataset=test_df,
    bearing_col="bearing",
    time_col="elapsed_sec",
)

print(
    f"Extracted {len(test_endpoints):,} endpoints "
    f"from {test_df['bearing'].nunique():,} bearings."
)

display(
    test_endpoints[[
        "bearing",
        "condition",
        "rpm",
        "load",
        "elapsed_sec",
    ]]
)

In [ ]:
# =============================================================================
# 5. Phase 1 — Endpoint Checkpoint
# =============================================================================

print("=" * 60)
print("Phase 1 — Endpoint Checkpoint")
print("=" * 60)

# -------------------------------------------------------------------------
# One endpoint per bearing
# -------------------------------------------------------------------------
n_bearings = test_df["bearing"].nunique()
n_endpoints = len(test_endpoints)

if n_endpoints != n_bearings:
    raise RuntimeError(
        "Endpoint count mismatch: "
        f"expected {n_bearings}, got {n_endpoints}."
    )

# -------------------------------------------------------------------------
# Bearing uniqueness
# -------------------------------------------------------------------------
if test_endpoints["bearing"].duplicated().any():
    raise RuntimeError(
        "Endpoint table contains duplicate bearings."
    )

# -------------------------------------------------------------------------
# Endpoint really is the maximum elapsed time
# -------------------------------------------------------------------------
expected_max_elapsed = (
    test_df
    .groupby("bearing")["elapsed_sec"]
    .max().sort_index()
)

actual_endpoint_elapsed = (
    test_endpoints
    .set_index("bearing")["elapsed_sec"]
    .sort_index()
)

if not expected_max_elapsed.equals(actual_endpoint_elapsed):
    raise RuntimeError(
        "Endpoint elapsed time does not match "
        "the maximum elapsed time of each bearing."
    )

# -------------------------------------------------------------------------
# Endpoint schema
# -------------------------------------------------------------------------
assert_required_columns(
    test_endpoints,
    [*META_COLUMNS, *reference_bundle.feature_columns],
)

print(f"Test bearings           : {n_bearings}")
print(f"Endpoint rows            : {n_endpoints}")
print("One endpoint per bearing : OK")
print("Maximum elapsed time     : OK")
print("Endpoint schema          : OK")
print("Phase 1 checkpoint passed.")

In [ ]:
# =============================================================================
# Phase 2 — Per-LOBO Test Prediction
# =============================================================================

print("=" * 60)
print("Phase 2 — Per-LOBO Test Prediction")
print("=" * 60)

# -------------------------------------------------------------------------
# Build LOBO-bearing → algorithm selection
# -------------------------------------------------------------------------
algorithm_by_bearing = {
    held_out_bearing: MODEL_SELECTION.algorithm_for(held_out_bearing)
    for held_out_bearing in reference_fold_bearings
}

print("LOBO algorithm selection")

for held_out_bearing in reference_fold_bearings:
    print(
        f"  {held_out_bearing:12s} → "
        f"{algorithm_by_bearing[held_out_bearing]}"
    )

# -------------------------------------------------------------------------
# Predict every Test endpoint with every selected LOBO model
# -------------------------------------------------------------------------
lobo_test_predictions, selected_algorithms = (
    predict_selected_lobo_test_endpoints(
        bundles=model_bundles,
        algorithm_by_bearing=algorithm_by_bearing,
        endpoints=test_endpoints,
    )
)

prediction_columns = [
    column for column in lobo_test_predictions.columns
    if column.startswith("predicted_RUL_norm_model_")
    and not column.endswith("_calibrated")
]

print(f"Test endpoints : {len(lobo_test_predictions):,}")
print(f"LOBO models    : {len(prediction_columns):,}")

display(
    lobo_test_predictions[[
        "bearing",
        "condition",
        "rpm",
        "load",
        "elapsed_sec",
        *prediction_columns,
    ]]
)

In [ ]:
model_selection_table = pd.DataFrame({
    "model_index": range(len(reference_fold_bearings)),
    "held_out_bearing": reference_fold_bearings,
    "algorithm": selected_algorithms,
})

display(model_selection_table)

In [ ]:
print("=" * 60)
print("Phase 2 — Prediction Checkpoint")
print("=" * 60)

expected_rows = len(test_endpoints)
expected_models = len(reference_fold_bearings)

if len(lobo_test_predictions) != expected_rows:
    raise RuntimeError(
        "Prediction row count mismatch: "
        f"expected {expected_rows}, "
        f"received {len(lobo_test_predictions)}."
    )

if len(prediction_columns) != expected_models:
    raise RuntimeError(
        "Prediction model count mismatch: "
        f"expected {expected_models}, "
        f"received {len(prediction_columns)}."
    )

if not lobo_test_predictions["bearing"].equals(test_endpoints["bearing"]):
    raise RuntimeError(
        "Prediction bearing order does not match Test endpoints."
    )

if not lobo_test_predictions["elapsed_sec"].equals(
    test_endpoints["elapsed_sec"]
):
    raise RuntimeError(
        "Prediction elapsed-time values do not match Test endpoints."
    )

# -------------------------------------------------------------------------
# Validate selected LOBO algorithm mapping
# -------------------------------------------------------------------------
if len(selected_algorithms) != expected_models:
    raise RuntimeError(
        "Selected algorithm count mismatch: "
        f"expected {expected_models}, "
        f"received {len(selected_algorithms)}."
    )

expected_selected_algorithms = [
    algorithm_by_bearing[bearing]
    for bearing in reference_fold_bearings
]

if selected_algorithms != expected_selected_algorithms:
    raise RuntimeError(
        "Selected algorithm order does not match LOBO model order."
    )

if model_selection_table["model_index"].tolist() != list(
    range(expected_models)
):
    raise RuntimeError("Model selection index order is invalid.")

# -------------------------------------------------------------------------
# Build and validate prediction matrix
# -------------------------------------------------------------------------
prediction_matrix = (
    lobo_test_predictions[prediction_columns]
    .to_numpy(dtype=float)
)

if prediction_matrix.shape != (expected_rows, expected_models):
    raise RuntimeError(
        "Prediction matrix shape mismatch: "
        f"expected {(expected_rows, expected_models)}, "
        f"received {prediction_matrix.shape}."
    )

if not pd.notna(prediction_matrix).all():
    raise RuntimeError("Prediction matrix contains missing values.")

if not np.isfinite(prediction_matrix).all():
    raise RuntimeError("Prediction matrix contains NaN or Inf values.")

print(f"Prediction Matrix shape : {prediction_matrix.shape}")
print("Endpoint alignment      : OK")
print("Model count             : OK")
print("Algorithm mapping       : OK")
print("Prediction validity     : OK")
print("Phase 2 checkpoint passed.")

In [ ]:
print("=" * 60)
print("Phase 3-1 — Raw Ensemble")
print("=" * 60)

# -------------------------------------------------------------------------
# 1. Ensemble configuration
# -------------------------------------------------------------------------
ensemble_config = EnsembleConfig(
    method=ensemble_method,
    weights=ensemble_weights,
)

# -------------------------------------------------------------------------
# 2. Raw ensemble
# -------------------------------------------------------------------------
raw_ensembled_rul_norm = ensemble_predictions(
    prediction_matrix=prediction_matrix,
    config=ensemble_config,
)

if not np.isfinite(raw_ensembled_rul_norm).all():
    raise RuntimeError(
        "Raw ensemble prediction contains NaN or Inf values."
    )

if raw_ensembled_rul_norm.ndim != 1:
    raise RuntimeError(
        "Raw ensemble prediction must be one-dimensional."
    )

if len(raw_ensembled_rul_norm) != len(test_endpoints):
    raise RuntimeError(
        "Raw ensemble prediction count mismatch: "
        f"expected {len(test_endpoints)}, "
        f"received {len(raw_ensembled_rul_norm)}."
    )

print(f"Ensemble method : {ensemble_config.method}")
print(f"Weights         : {ensemble_config.weights}")
print(f"Prediction count: {len(raw_ensembled_rul_norm)}")
print("Raw ensemble    : OK")

In [ ]:
print("=" * 60)
print("Phase 3-2 — Model-specific Calibration")
print("=" * 60)

# -------------------------------------------------------------------------
# 1. Calibration availability
# -------------------------------------------------------------------------
calibrated_prediction_matrix = None
calibrated_ensembled_rul_norm = None
calibrated_prediction_columns = []

calibration_results_by_algorithm = {}
calibration_parameters_by_algorithm = {}

selected_calibration_paths = {
    algorithm: MODEL_PATHS[algorithm]["calibration"]
    for algorithm in set(selected_algorithms)
}

calibration_available = (
    CALIBRATION_METHOD is not None
    and all(
        path.exists()
        for path in selected_calibration_paths.values()
    )
)

if not calibration_available:
    print(
        "Calibration results are not fully available. "
        "Raw model predictions will be used."
    )

else:
    # ---------------------------------------------------------------------
    # 2. Load calibration results for selected algorithms
    # ---------------------------------------------------------------------
    for algorithm, path in selected_calibration_paths.items():
        with path.open("r", encoding="utf-8") as file:
            calibration_results = json.load(file)

        if not isinstance(calibration_results, dict):
            raise TypeError(
                f"Calibration results for {algorithm!r} "
                "must be a JSON object."
            )

        calibration_file_method = calibration_results.get("method")
        parameters_by_method = calibration_results.get("parameters")

        if not isinstance(parameters_by_method, dict):
            raise ValueError(
                f"Calibration results for {algorithm!r} "
                "must contain a 'parameters' object."
            )

        if CALIBRATION_METHOD not in parameters_by_method:
            raise ValueError(
                f"Calibration method {CALIBRATION_METHOD!r} is not "
                f"available for algorithm {algorithm!r}."
            )

        if calibration_file_method not in (
            CALIBRATION_METHOD, "both"):
            raise ValueError(
                "Calibration method mismatch: "
                f"algorithm={algorithm!r}, "
                f"file={calibration_file_method!r}, "
                f"requested={CALIBRATION_METHOD!r}."
            )

        calibration_parameters = (
            parameters_by_method[CALIBRATION_METHOD]
        )

        if not isinstance(calibration_parameters, dict):
            raise ValueError(
                f"Calibration parameters for {algorithm!r} "
                f"and method {CALIBRATION_METHOD!r} "
                "must be an object."
            )

        calibration_results_by_algorithm[algorithm] = calibration_results
        calibration_parameters_by_algorithm[algorithm] = (
            calibration_parameters
        )

    # ---------------------------------------------------------------------
    # 3. Validate calibration model indices
    # ---------------------------------------------------------------------
    expected_models = len(selected_algorithms)

    for model_index, algorithm in enumerate(selected_algorithms):
        calibration_parameters = (
            calibration_parameters_by_algorithm[algorithm]
        )

        expected_indices = {
            str(index) for index in range(
                len(model_bundles[algorithm].models)
            )
        }

        if set(calibration_parameters) != expected_indices:
            raise ValueError(
                "Calibration model index set does not match Model Bundle: "
                f"algorithm={algorithm!r}, "
                f"expected={sorted(expected_indices)}, "
                f"received={sorted(calibration_parameters)}."
            )

    # ---------------------------------------------------------------------
    # 4. Apply calibration per selected LOBO model
    # ---------------------------------------------------------------------
    calibrated_prediction_matrix = prediction_matrix.copy()

    parameter_name = (
        "gamma" if CALIBRATION_METHOD == "power"
        else "a"
    )

    for model_index, algorithm in enumerate(selected_algorithms):
        calibration_parameters = (
            calibration_parameters_by_algorithm[algorithm]
        )

        parameter_entry = calibration_parameters[str(model_index)]

        if not isinstance(parameter_entry, dict):
            raise ValueError(
                f"Calibration entry for {algorithm!r}, "
                f"model_{model_index} must be an object."
            )

        if parameter_name not in parameter_entry:
            raise ValueError(
                f"Calibration entry for {algorithm!r}, "
                f"model_{model_index} does not contain "
                f"{parameter_name!r}."
            )

        parameter = float(parameter_entry[parameter_name])

        calibrated_prediction_matrix[:, model_index] = (
            apply_rul_calibration(
                prediction_matrix[:, model_index],
                method=CALIBRATION_METHOD,
                parameter=parameter,
            )
        )

    # ---------------------------------------------------------------------
    # 5. Validate calibrated prediction matrix
    # ---------------------------------------------------------------------
    if not np.isfinite(calibrated_prediction_matrix).all():
        raise RuntimeError(
            "Calibrated prediction matrix contains NaN or Inf values."
        )

    if not (
        (calibrated_prediction_matrix >= 0.0)
        & (calibrated_prediction_matrix <= 1.0)
    ).all():
        raise RuntimeError(
            "Calibrated prediction matrix violates the [0, 1] domain."
        )

    # ---------------------------------------------------------------------
    # 6. Store calibrated model predictions
    # ---------------------------------------------------------------------
    for model_index in range(expected_models):
        column = (
            "predicted_RUL_norm_model_"
            f"{model_index}_calibrated"
        )

        lobo_test_predictions[column] = (
            calibrated_prediction_matrix[:, model_index]
        )

        calibrated_prediction_columns.append(column)

    # ---------------------------------------------------------------------
    # 7. Calibrated ensemble
    # ---------------------------------------------------------------------
    calibrated_ensembled_rul_norm = ensemble_predictions(
        prediction_matrix=calibrated_prediction_matrix,
        config=ensemble_config,
    )

    if not np.isfinite(calibrated_ensembled_rul_norm).all():
        raise RuntimeError(
            "Calibrated ensemble prediction contains "
            "NaN or Inf values."
        )

    print(f"Calibration method : {CALIBRATION_METHOD}")
    print(f"Calibrated models  : {len(selected_algorithms)}")
    print(
        "Calibration configs: "
        f"{', '.join(sorted(selected_calibration_paths))}"
    )
    print("Calibration        : OK")
    print("Calibrated ensemble: OK")

In [ ]:
print("=" * 60)
print("Phase 3-3 — Active Result Selection")
print("=" * 60)

# -------------------------------------------------------------------------
# 1. Select active final prediction
# -------------------------------------------------------------------------
if SUBMISSION_USE_CALIBRATION and calibration_available:
    ensembled_rul_norm = calibrated_ensembled_rul_norm
    prediction_matrix_for_ensemble = calibrated_prediction_matrix
    active_prediction_label = "calibrated"

else:
    ensembled_rul_norm = raw_ensembled_rul_norm
    prediction_matrix_for_ensemble = prediction_matrix
    active_prediction_label = "raw"

# -------------------------------------------------------------------------
# 2. Store ensemble predictions
# -------------------------------------------------------------------------
lobo_test_predictions["predicted_RUL_norm_raw_ensemble"] = (
    raw_ensembled_rul_norm
)

if calibrated_ensembled_rul_norm is not None:
    lobo_test_predictions[
        "predicted_RUL_norm_calibrated_ensemble"
    ] = calibrated_ensembled_rul_norm

lobo_test_predictions["predicted_RUL_norm"] = ensembled_rul_norm

# -------------------------------------------------------------------------
# 3. Display result
# -------------------------------------------------------------------------
display_columns = [
    "bearing",
    "elapsed_sec",
    *prediction_columns,
]

if calibrated_prediction_columns:
    display_columns.extend(calibrated_prediction_columns)

display_columns.append("predicted_RUL_norm_raw_ensemble")

if calibrated_ensembled_rul_norm is not None:
    display_columns.append("predicted_RUL_norm_calibrated_ensemble")

display_columns.append("predicted_RUL_norm")

print(f"Ensemble method : {ensemble_config.method}")
print(f"Weights         : {ensemble_config.weights}")
print(
    "Calibration     : "
    f"{CALIBRATION_METHOD if calibration_available else 'unavailable'}"
)
print(f"Active result   : {active_prediction_label}")

display(lobo_test_predictions[display_columns])

In [ ]:
print("=" * 60)
print("Phase 3 — Ensemble Checkpoint")
print("=" * 60)

if len(ensembled_rul_norm) != len(test_endpoints):
    raise RuntimeError(
        "Ensemble prediction count mismatch: "
        f"expected {len(test_endpoints)}, "
        f"received {len(ensembled_rul_norm)}."
    )

if ensembled_rul_norm.ndim != 1:
    raise RuntimeError("Ensemble prediction must be one-dimensional.")

if not np.isfinite(ensembled_rul_norm).all():
    raise RuntimeError("Ensemble prediction contains NaN or Inf values.")

if not np.isfinite(
    lobo_test_predictions["predicted_RUL_norm"].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Stored ensemble prediction contains NaN or Inf values."
    )

if not np.allclose(
    lobo_test_predictions["predicted_RUL_norm"].to_numpy(dtype=float),
    ensembled_rul_norm,
):
    raise RuntimeError(
        "Stored ensemble prediction does not match "
        "the active ensemble result."
    )

# Arithmetic sanity check is retained only for arithmetic ensemble mode.
if ensemble_method == "arithmetic":
    expected_arithmetic = np.mean(
        prediction_matrix_for_ensemble,
        axis=1,
    )

    if not np.allclose(
        ensembled_rul_norm,
        expected_arithmetic,
    ):
        raise RuntimeError(
            "Arithmetic ensemble result is incorrect."
        )

# Validate raw ensemble independently.
if not np.isfinite(raw_ensembled_rul_norm).all():
    raise RuntimeError(
        "Raw ensemble prediction contains NaN or Inf values."
    )

if ensemble_method == "arithmetic":
    expected_raw_arithmetic = np.mean(
        prediction_matrix,
        axis=1,
    )

    if not np.allclose(
        raw_ensembled_rul_norm,
        expected_raw_arithmetic,
    ):
        raise RuntimeError(
            "Raw arithmetic ensemble result is incorrect."
        )

if calibrated_ensembled_rul_norm is not None:
    if not np.isfinite(calibrated_ensembled_rul_norm).all():
        raise RuntimeError(
            "Calibrated ensemble prediction contains NaN or Inf values."
        )

print(f"Ensemble samples : {len(ensembled_rul_norm)}")
print(f"Ensemble method  : {ensemble_config.method}")
print(f"Active result    : {active_prediction_label}")
print("Prediction count : OK")
print("Prediction validity : OK")
print("Stored result    : OK")
print("Ensemble calculation : OK")
print("Phase 3 checkpoint passed.")

In [ ]:
print("=" * 60)
print("Phase 4 — RUL Reconstruction")
print("=" * 60)

# -------------------------------------------------------------------------
# Raw result
# -------------------------------------------------------------------------
raw_lobo_predictions = lobo_test_predictions[
    ["bearing", "elapsed_sec"]
].copy()
raw_lobo_predictions["predicted_RUL_norm"] = raw_ensembled_rul_norm

raw_test_predictions_sec = reconstruct_test_rul_sec_from_norm(
    df_predictions=raw_lobo_predictions,
    time_col="elapsed_sec",
    pred_norm_col="predicted_RUL_norm",
)

# -------------------------------------------------------------------------
# Calibrated result, when available
# -------------------------------------------------------------------------
calibrated_test_predictions_sec = None

if calibrated_ensembled_rul_norm is not None:
    calibrated_lobo_predictions = lobo_test_predictions[
        ["bearing", "elapsed_sec"]
    ].copy()
    calibrated_lobo_predictions["predicted_RUL_norm"] = (
        calibrated_ensembled_rul_norm
    )

    calibrated_test_predictions_sec = (
        reconstruct_test_rul_sec_from_norm(
            df_predictions=calibrated_lobo_predictions,
            time_col="elapsed_sec",
            pred_norm_col="predicted_RUL_norm",
        )
    )

# Active result used for the submission artifact.
test_predictions_sec = (
    calibrated_test_predictions_sec
    if active_prediction_label == "calibrated"
    else raw_test_predictions_sec
)

comparison_columns = {
    "bearing": test_predictions_sec["bearing"],
    "raw_RUL_sec": raw_test_predictions_sec["predicted_RUL_sec"],
}

if calibrated_test_predictions_sec is not None:
    comparison_columns["calibrated_RUL_sec"] = (
        calibrated_test_predictions_sec["predicted_RUL_sec"]
    )

test_rul_comparison = pd.DataFrame(comparison_columns)

print("Raw and calibrated results are preserved separately.")
display(test_rul_comparison)

In [ ]:
print("=" * 60)
print("Phase 4 — Reconstruction Checkpoint")
print("=" * 60)

def _validate_reconstructed_predictions(
    predictions: pd.DataFrame,
    expected_endpoints: pd.DataFrame,
    label: str,
):
    if len(predictions) != len(expected_endpoints):
        raise RuntimeError(
            f"{label} reconstruction row count mismatch: "
            f"expected {len(expected_endpoints)}, "
            f"received {len(predictions)}."
        )

    if "predicted_RUL_sec" not in predictions.columns:
        raise RuntimeError(
            f"{label} reconstruction is missing 'predicted_RUL_sec'."
        )

    pred_norm = predictions["predicted_RUL_norm"].to_numpy(dtype=float)
    elapsed_sec = predictions["elapsed_sec"].to_numpy(dtype=float)
    pred_rul_sec = predictions["predicted_RUL_sec"].to_numpy(dtype=float)

    if not np.isfinite(pred_rul_sec).all():
        raise RuntimeError(
            f"{label} reconstructed RUL_sec contains NaN or Inf values."
        )

    if not ((pred_norm >= 0) & (pred_norm < 1)).all():
        raise RuntimeError(
            f"{label} predicted_RUL_norm violates the reconstruction contract."
        )

    if (elapsed_sec < 0).any():
        raise RuntimeError(
            f"{label} elapsed_sec contains negative values."
        )

    expected_rul_sec = elapsed_sec * pred_norm / (1.0 - pred_norm)

    if not np.allclose(pred_rul_sec, expected_rul_sec):
        raise RuntimeError(f"{label} RUL_sec reconstruction formula mismatch.")

    if (pred_rul_sec < 0).any():
        raise RuntimeError(f"{label} reconstructed RUL_sec contains negative values.")

    if not predictions["bearing"].equals(expected_endpoints["bearing"]):
        raise RuntimeError(
            f"{label} reconstructed prediction bearing order "
            "does not match Test endpoints."
        )

    recovered_norm = pred_rul_sec / (elapsed_sec + pred_rul_sec)

    if not np.allclose(recovered_norm, pred_norm):
        raise RuntimeError(f"{label} RUL_norm inverse relation is inconsistent.")

_validate_reconstructed_predictions(
    raw_test_predictions_sec,
    test_endpoints,
    "Raw",
)

if calibrated_test_predictions_sec is not None:
    _validate_reconstructed_predictions(
        calibrated_test_predictions_sec,
        test_endpoints,
        "Calibrated",
    )

print(f"Prediction rows        : {len(test_predictions_sec)}")
print("Raw reconstruction     : OK")
print(
    "Calibrated reconstruction: "
    f"{'OK' if calibrated_test_predictions_sec is not None else 'skipped'}"
)
print("Phase 4 checkpoint passed.")


In [ ]:
print("=" * 60)
print("Phase 5 — Optional Test Evaluation")
print("=" * 60)

test_evaluation_df = None
test_metrics = None

TEST_GROUND_TRUTH = {
    "Bearing1_3": 5730.0,
    "Bearing1_4": 339.0,
    "Bearing1_5": 1610.0,
    "Bearing1_6": 1460.0,
    "Bearing1_7": 7570.0,
    "Bearing2_3": 7530.0,
    "Bearing2_4": 1390.0,
    "Bearing2_5": 3090.0,
    "Bearing2_6": 1290.0,
    "Bearing2_7": 580.0,
    "Bearing3_3": 820.0,
}

if TEST_GROUND_TRUTH is None:
    print("External Test Ground Truth is not provided.")
    print("Phase 5 evaluation will be skipped.")

else:
    if not isinstance(TEST_GROUND_TRUTH, dict):
        raise TypeError("TEST_GROUND_TRUTH must be a dict or None.")

    if not TEST_GROUND_TRUTH:
        raise ValueError(
            "TEST_GROUND_TRUTH must not be empty. "
            "Use None to skip evaluation."
        )

    ground_truth_df = pd.DataFrame(
        TEST_GROUND_TRUTH.items(),
        columns=["bearing", "actual_RUL_sec"],
    )

    if ground_truth_df["bearing"].isna().any():
        raise ValueError("TEST_GROUND_TRUTH contains a missing bearing.")

    if ground_truth_df["bearing"].duplicated().any():
        duplicated_bearings = (
            ground_truth_df.loc[
                ground_truth_df["bearing"].duplicated(keep=False),
                "bearing",
            ].unique().tolist()
        )

        raise ValueError(
            "TEST_GROUND_TRUTH must contain exactly one "
            "value per bearing. "
            f"Duplicated bearings: {duplicated_bearings}"
        )

    if not pd.api.types.is_numeric_dtype(ground_truth_df["actual_RUL_sec"]):
        raise TypeError("'actual_RUL_sec' must be numeric.")

    actual_rul = ground_truth_df["actual_RUL_sec"].to_numpy(dtype=float)

    if not np.isfinite(actual_rul).all():
        raise ValueError("'actual_RUL_sec' contains NaN or Inf values.")

    if (actual_rul < 0).any():
        raise ValueError("'actual_RUL_sec' must contain non-negative values.")

    # ---------------------------------------------------------------------
    # Evaluate raw result
    # ---------------------------------------------------------------------
    raw_evaluation_df = raw_test_predictions_sec[[
        "bearing",
        "elapsed_sec",
        "predicted_RUL_norm",
        "predicted_RUL_sec",
    ]].merge(
        ground_truth_df,
        on="bearing",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )

    if (raw_evaluation_df["_merge"] != "both").any():
        raise ValueError(
            "Raw Test prediction bearings and Ground Truth bearings "
            "do not match."
        )

    raw_evaluation_df = (
        raw_evaluation_df
        .drop(columns="_merge")
        .sort_values("bearing")
        .reset_index(drop=True)
    )

    raw_metrics = evaluate_prediction_table(
        raw_evaluation_df,
        actual_col="actual_RUL_sec",
        predicted_col="predicted_RUL_sec",
    )

    # ---------------------------------------------------------------------
    # Evaluate calibrated result when available
    # ---------------------------------------------------------------------
    calibrated_evaluation_df = None
    calibrated_metrics = None

    if calibrated_test_predictions_sec is not None:
        calibrated_evaluation_df = (
            calibrated_test_predictions_sec[[
                "bearing",
                "elapsed_sec",
                "predicted_RUL_norm",
                "predicted_RUL_sec",
            ]].merge(
                ground_truth_df,
                on="bearing",
                how="outer",
                validate="one_to_one",
                indicator=True,
            )
        )

        if (calibrated_evaluation_df["_merge"] != "both").any():
            raise ValueError(
                "Calibrated Test prediction bearings and Ground Truth "
                "bearings do not match."
            )

        calibrated_evaluation_df = (
            calibrated_evaluation_df
            .drop(columns="_merge")
            .sort_values("bearing")
            .reset_index(drop=True)
        )

        calibrated_metrics = evaluate_prediction_table(
            calibrated_evaluation_df,
            actual_col="actual_RUL_sec",
            predicted_col="predicted_RUL_sec",
        )

    # ---------------------------------------------------------------------
    # Combined evaluation table
    # ---------------------------------------------------------------------
    test_evaluation_df = raw_evaluation_df[
        ["bearing", "actual_RUL_sec", "predicted_RUL_sec"]
    ].rename(columns={"predicted_RUL_sec": "raw_RUL_sec"})

    if calibrated_evaluation_df is not None:
        test_evaluation_df = test_evaluation_df.merge(
            calibrated_evaluation_df[
                ["bearing", "predicted_RUL_sec"]
            ].rename(columns={"predicted_RUL_sec": "calibrated_RUL_sec"}),
            on="bearing",
            how="left",
            validate="one_to_one",
        )

    test_metrics = {"raw": raw_metrics}

    if calibrated_metrics is not None:
        test_metrics["calibrated"] = calibrated_metrics

    print("Test Ground Truth loaded from Notebook.")
    print(f"Ground Truth bearings : {len(ground_truth_df)}")
    print(f"Evaluation rows       : {len(test_evaluation_df)}")

    print("\nTest Metrics")
    print("\nRaw")
    for metric_name, metric_value in raw_metrics.items():
        print(f"{metric_name}: {metric_value}")

    if calibrated_metrics is not None:
        print("\nCalibrated")
        for metric_name, metric_value in calibrated_metrics.items():
            print(f"{metric_name}: {metric_value}")

    display(test_evaluation_df)

In [ ]:
print("=" * 60)
print("Phase 5 — Evaluation Checkpoint")
print("=" * 60)

if TEST_GROUND_TRUTH is None:
    print("Evaluation skipped.")
    print("Phase 5 checkpoint passed (evaluation skipped).")

else:
    if test_evaluation_df is None:
        raise RuntimeError(
            "Evaluation was expected but test_evaluation_df is None."
        )

    if test_metrics is None:
        raise RuntimeError("Evaluation was expected but test_metrics is None.")

    expected_bearings = set(test_endpoints["bearing"])
    actual_bearings = set(test_evaluation_df["bearing"])

    if actual_bearings != expected_bearings:
        raise RuntimeError(
            "Evaluation bearing set does not match Test prediction "
            "bearing set."
        )

    if len(test_evaluation_df) != len(test_endpoints):
        raise RuntimeError(
            "Evaluation row count does not match Test prediction "
            "row count."
        )

    if not np.isfinite(
        test_evaluation_df["actual_RUL_sec"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError("Evaluation Ground Truth contains non-finite values.")

    required_evaluation_metrics = {
        "RMSE",
        "MAE",
        "SMAPE",
        "R2",
        "PHM_Score",
    }

    if set(test_metrics["raw"]) != required_evaluation_metrics:
        raise RuntimeError(
            "Unexpected Raw Test metric set: "
            f"{set(test_metrics['raw'])}"
        )

    if "calibrated" in test_metrics:
        if set(test_metrics["calibrated"]) != required_evaluation_metrics:
            raise RuntimeError(
                "Unexpected Calibrated Test metric set: "
                f"{set(test_metrics['calibrated'])}"
            )

        if "calibrated_RUL_sec" not in test_evaluation_df.columns:
            raise RuntimeError(
                "Calibrated metrics exist but calibrated predictions "
                "are missing from the evaluation table."
            )

        if not np.isfinite(
            test_evaluation_df["calibrated_RUL_sec"].to_numpy(dtype=float)
        ).all():
            raise RuntimeError(
                "Calibrated evaluation predictions contain non-finite values."
            )

    if not np.isfinite(
        test_evaluation_df["raw_RUL_sec"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            "Raw evaluation predictions contain non-finite values."
        )

    print(f"Evaluation bearings : {len(test_evaluation_df)}")
    print("Bearing alignment   : OK")
    print("Row count           : OK")
    print("Actual RUL validity : OK")
    print("Raw prediction      : OK")
    print(
        "Calibrated prediction: "
        f"{'OK' if 'calibrated' in test_metrics else 'skipped'}"
    )
    print("Metric set          : OK")
    print("Phase 5 checkpoint passed.")

In [ ]:
print("=" * 60)
print("Phase 6 — Final Artifact Saving")
print("=" * 60)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# 1. LOBO prediction artifact
# -------------------------------------------------------------------------
lobo_test_predictions.to_csv(LOBO_PREDICTIONS_PATH, index=False)

# -------------------------------------------------------------------------
# 2. Raw / calibrated comparison artifact
# -------------------------------------------------------------------------
test_rul_comparison.to_csv(COMPARISON_PREDICTIONS_PATH, index=False)

# -------------------------------------------------------------------------
# 3. Submission artifact
# -------------------------------------------------------------------------
submission_rul_sec = (
    test_predictions_sec["predicted_RUL_sec"].to_numpy(dtype=float)
)

submission_predictions = pd.DataFrame({
    "bearing": test_predictions_sec["bearing"],
    "RUL_sec": submission_rul_sec,
})

submission_predictions.to_csv(SUBMISSION_PATH, index=False)

# -------------------------------------------------------------------------
# 4. Ensemble configuration
# -------------------------------------------------------------------------
ensemble_config_payload = {
    "method": ensemble_config.method,
    "weights": ensemble_config.weights,
    "model_selection": [
        {
            "model_index": int(row["model_index"]),
            "held_out_bearing": row["held_out_bearing"],
            "algorithm": row["algorithm"],
        } for _, row in model_selection_table.iterrows()
    ],
    "calibration": {
        "enabled": calibrated_ensembled_rul_norm is not None,
        "method": (
            CALIBRATION_METHOD
            if calibrated_ensembled_rul_norm is not None
            else None
        ),
    },
}

with ENSEMBLE_CONFIG_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        ensemble_config_payload,
        file,
        indent=2,
        ensure_ascii=False,
    )

# -------------------------------------------------------------------------
# 5. Optional evaluation metrics
# -------------------------------------------------------------------------
if test_metrics is not None:
    with TEST_METRICS_PATH.open("w", encoding="utf-8") as file:
        json.dump(
            test_metrics,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print(f"Test metrics saved      : {TEST_METRICS_PATH}")
else:
    if TEST_METRICS_PATH.exists():
        TEST_METRICS_PATH.unlink()

    print("Test metrics             : skipped")

print(f"LOBO predictions saved   : {LOBO_PREDICTIONS_PATH}")
print(f"Comparison saved         : {COMPARISON_PREDICTIONS_PATH}")
print(f"Submission saved         : {SUBMISSION_PATH}")
print(f"Ensemble config saved    : {ENSEMBLE_CONFIG_PATH}")
print(f"Submission result        : {active_prediction_label}")

In [ ]:
print("=" * 60)
print("Phase 6 — Artifact Checkpoint")
print("=" * 60)

# -------------------------------------------------------------------------
# 1. File existence
# -------------------------------------------------------------------------
required_paths = [
    LOBO_PREDICTIONS_PATH,
    COMPARISON_PREDICTIONS_PATH,
    SUBMISSION_PATH,
    ENSEMBLE_CONFIG_PATH,
]

for path in required_paths:
    if not path.exists():
        raise RuntimeError(f"Required artifact was not created: {path}")

if test_metrics is not None:
    if not TEST_METRICS_PATH.exists():
        raise RuntimeError(
            "Test metrics were produced but "
            "test_metrics.json was not created."
        )
else:
    if TEST_METRICS_PATH.exists():
        raise RuntimeError(
            "test_metrics.json exists although "
            "Test evaluation was skipped."
        )

print("Required artifact files : OK")

# -------------------------------------------------------------------------
# 2. Reload LOBO predictions
# -------------------------------------------------------------------------
saved_lobo_predictions = pd.read_csv(LOBO_PREDICTIONS_PATH)

if len(saved_lobo_predictions) != len(lobo_test_predictions):
    raise RuntimeError("Saved LOBO prediction row count mismatch.")

if list(saved_lobo_predictions.columns) != list(
    lobo_test_predictions.columns
):
    raise RuntimeError("Saved LOBO prediction schema mismatch.")

print("LOBO prediction artifact : OK")

# -------------------------------------------------------------------------
# 3. Reload comparison predictions
# -------------------------------------------------------------------------
saved_comparison = pd.read_csv(COMPARISON_PREDICTIONS_PATH)

if len(saved_comparison) != len(test_rul_comparison):
    raise RuntimeError("Saved comparison row count mismatch.")

if list(saved_comparison.columns) != list(test_rul_comparison.columns):
    raise RuntimeError("Saved comparison schema mismatch.")

for column in test_rul_comparison.columns:
    if column == "bearing":
        if not saved_comparison[column].equals(
            test_rul_comparison[column]
        ):
            raise RuntimeError(
                f"Saved '{column}' values do not match."
            )
    else:
        if not np.allclose(
            saved_comparison[column].to_numpy(dtype=float),
            test_rul_comparison[column].to_numpy(dtype=float),
        ):
            raise RuntimeError(
                f"Saved '{column}' values do not match."
            )

print("Comparison artifact      : OK")

# -------------------------------------------------------------------------
# 4. Reload submission
# -------------------------------------------------------------------------
saved_submission = pd.read_csv(SUBMISSION_PATH)

if list(saved_submission.columns) != ["bearing", "RUL_sec"]:
    raise RuntimeError("Submission schema mismatch.")

if len(saved_submission) != len(submission_predictions):
    raise RuntimeError("Saved submission row count mismatch.")

if not saved_submission["bearing"].equals(
    submission_predictions["bearing"]
):
    raise RuntimeError("Saved submission bearing values do not match.")

if not np.allclose(
    saved_submission["RUL_sec"].to_numpy(dtype=float),
    submission_predictions["RUL_sec"].to_numpy(dtype=float),
):
    raise RuntimeError("Saved submission RUL values do not match.")

print("Submission artifact      : OK")

# -------------------------------------------------------------------------
# 5. Reload EnsembleConfig
# -------------------------------------------------------------------------

with ENSEMBLE_CONFIG_PATH.open("r", encoding="utf-8") as file:
    saved_ensemble_config = json.load(file)

if saved_ensemble_config != ensemble_config_payload:
    raise RuntimeError(
        "Saved EnsembleConfig does not match "
        "the active configuration."
    )

print("Ensemble config artifact : OK")

# -------------------------------------------------------------------------
# 6. Reload metrics when evaluation exists
# -------------------------------------------------------------------------

if test_metrics is not None:
    with TEST_METRICS_PATH.open("r", encoding="utf-8") as file:
        saved_metrics = json.load(file)

    if saved_metrics != test_metrics:
        raise RuntimeError(
            "Saved Test metrics do not match "
            "the active evaluation result."
        )

    print("Test metrics artifact    : OK")

print(f"Output directory         : {OUTPUT_DIR}")
print("Phase 6 checkpoint passed.")

In [ ]:
print("=" * 60)
print("Stage 06 — Final Integration Check")
print("=" * 60)

print(f"Test bearings       : {len(test_endpoints)}")
print(f"LOBO models         : {prediction_matrix.shape[1]}")
print(f"Prediction matrix   : {prediction_matrix.shape}")
print(f"Ensemble method     : {ensemble_config.method}")
print(f"Calibration         : {'available' if calibration_available else 'unavailable'}")
print(f"Active result       : {active_prediction_label}")
print(f"Comparison rows     : {len(test_rul_comparison)}")
print(f"Submission rows     : {len(submission_predictions)}")
print(
    "Evaluation          : "
    f"{'performed' if test_metrics is not None else 'skipped'}"
)
print(f"Output directory    : {OUTPUT_DIR}")

print()
print("Stage 06 integration check passed.")